## GHG Emissions Data - Gold Layer

## Objective
Transform, unpivot, and restructure wide Silver GHG emissions data from silver.silver_ghg_emissions into a clean, normalized, long-format Gold Delta table (gold.ghg_emissions) optimized for Power BI reporting and business analytics.

## Data Flow
silver.silver_ghg_emissions → Spark SQL Unpivot Transformation → PySpark DataFrame → gold.ghg_emissions

## Source
The underlying data comes from the Paris OpenData GHG Emissions Dataset (Bilan Carbone - Paris OpenData API).

## Input
Silver Delta table: silver.silver_ghg_emissions

## Output
Gold Delta table: gold.ghg_emissions

## Gold Layer Principle
The Gold layer provides business-level aggregations and normalized data structures tailored for downstream Business Intelligence (BI) tools like Power BI and Tableau. In this transformation, wide-format Silver data (where sub-sectors exist as individual columns across 8 time-series rows) is unpivoted into a long-format Gold table (80 rows). This structure decouples metrics from schema definitions, eliminates double-counting, and enables dynamic slicing by emission scope and sector.

## Processing Steps
1. **Audit Source:** Inspect silver.silver_ghg_emissions columns and row count.
2. **Unpivot Sub-Sectors:** Apply UNION ALL to map detail columns into emission_scope, sector, and emissions_mt_co2.
3. **Exclude Macro Totals:** Omit ghg_major_sectors columns to prevent double-counting.
4. **Save Delta Table:** Execute SQL query and overwrite gold.ghg_emissions.
5. **Validate Output:** Confirm target table contains exactly 80 rows.

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
%sql
-- Check the data 
SELECT * 
FROM workspace.silver.silver_ghg_emissions;

In [0]:
query = """
SELECT year, 'Local' AS emission_scope, 'Residential' AS sector, ghg_local_details_residential AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Local' AS emission_scope, 'Commercial' AS sector, ghg_local_details_commercial AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Local' AS emission_scope, 'Industry' AS sector, ghg_local_details_industry AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Local' AS emission_scope, 'Transport' AS sector, ghg_local_details_transport AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Local' AS emission_scope, 'Waste' AS sector, ghg_local_details_waste AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Outside Paris' AS emission_scope, 'Construction & Materials' AS sector, ghg_outside_paris_details_construction_materials AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Outside Paris' AS emission_scope, 'Transport' AS sector, ghg_outside_paris_details_transport AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Outside Paris' AS emission_scope, 'Air Transport' AS sector, ghg_outside_paris_details_air_transport AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Outside Paris' AS emission_scope, 'Food' AS sector, ghg_outside_paris_details_food AS emissions_mt_co2 FROM silver.silver_ghg_emissions
UNION ALL
SELECT year, 'Outside Paris' AS emission_scope, 'Upstream Energy' AS sector, ghg_outside_paris_details_upstream_energy AS emissions_mt_co2 FROM silver.silver_ghg_emissions
"""

# Execute SQL query and write to Gold Delta table
df_gold_ghg = spark.sql(query)

In [0]:
df_gold_ghg.display()

# WRITING GOLD TABLE

In [0]:
df_gold_ghg\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_ghg_emissions")

# CHECKING THE GOLD TABLE

In [0]:
%sql 
SELECT *
FROM workspace.gold.gold_ghg_emissions
LIMIT 10 ;